Student Admission Decision System

In [8]:
def get_valid_input(prompt, valid_options=None, input_type=str, min_val=None, max_val=None):
    """
    Generic input validator with type checking and range validation.
    
    Args:
        prompt: Display prompt for user
        valid_options: List/tuple of acceptable values (case-insensitive for strings)
        input_type: Expected type (str, int, float)
        min_val: Minimum acceptable value (for numeric inputs)
        max_val: Maximum acceptable value (for numeric inputs)
    
    Returns:
        Validated input of specified type
    """
    while True:
        try:
            user_input = input(prompt).strip()
            
            if not user_input:
                print("Error: Input cannot be empty. Please try again.")
                continue
            
            # Type conversion
            if input_type == int:
                value = int(user_input)
            elif input_type == float:
                value = float(user_input)
            else:
                value = user_input.lower() if valid_options else user_input
            
            # Range validation for numeric inputs
            if input_type in (int, float):
                if min_val is not None and value < min_val:
                    print(f"Error: Value must be at least {min_val}. Please try again.")
                    continue
                if max_val is not None and value > max_val:
                    print(f"Error: Value must be at most {max_val}. Please try again.")
                    continue
            
            # Option validation
            if valid_options:
                valid_lower = [str(opt).lower() for opt in valid_options]
                if str(value).lower() not in valid_lower:
                    print(f"Error: Invalid input. Accepted values: {', '.join(valid_options)}")
                    continue
                # Return original case for valid options
                idx = valid_lower.index(str(value).lower())
                return valid_options[idx]
            
            return value
            
        except ValueError:
            print(f"Error: Invalid input type. Expected {input_type.__name__}. Please try again.")


def calculate_admission_decision(entrance_score, gpa, has_recommendation, 
                                category, extracurricular_score):
    """
    Evaluate student admission based on university criteria.
    
    Returns:
        dict: Contains 'status', 'reason', 'effective_score', 'bonuses_applied'
    """
    
    # Category thresholds
    CATEGORY_CUTOFFS = {
        'general': 75,
        'obc': 65,
        'sc_st': 55
    }
    
    MIN_GPA = 7.0
    SCHOLARSHIP_THRESHOLD = 95
    
    result = {
        'effective_score': entrance_score,
        'bonuses_applied': [],
        'status': None,
        'reason': None
    }
    
    # Merit Rule: Auto-admit with scholarship if entrance score >= 95
    if entrance_score >= SCHOLARSHIP_THRESHOLD:
        result['status'] = "ADMITTED (Scholarship)"
        result['reason'] = f"Auto-admitted: Entrance score {entrance_score} ≥ {SCHOLARSHIP_THRESHOLD} (Merit Rule)"
        return result
    
    # Calculate bonuses
    bonus_total = 0
    
    # Recommendation bonus: +5 points
    if has_recommendation.lower() == 'yes':
        bonus_total += 5
        result['bonuses_applied'].append("Recommendation (+5)")
    
    # Extracurricular bonus: +3 points if score > 8
    if extracurricular_score > 8:
        bonus_total += 3
        result['bonuses_applied'].append("Extracurricular (+3)")
    
    result['effective_score'] = entrance_score + bonus_total
    
    # Check minimum GPA requirement (universal for all categories)
    if gpa < MIN_GPA:
        result['status'] = "REJECTED"
        result['reason'] = f"GPA requirement not met: {gpa} < {MIN_GPA} (required for all categories)"
        return result
    
    # Check category-specific entrance cutoff
    required_cutoff = CATEGORY_CUTOFFS[category]
    
    if result['effective_score'] >= required_cutoff:
        result['status'] = "ADMITTED (Regular)"
        result['reason'] = (f"Meets {category.upper()} cutoff "
                          f"({result['effective_score']} ≥ {required_cutoff}) "
                          f"and GPA requirement ({gpa} ≥ {MIN_GPA})")
    else:
        result['status'] = "WAITLISTED"
        result['reason'] = (f"Below {category.upper()} cutoff: "
                          f"Effective score {result['effective_score']} < {required_cutoff} "
                          f"(required: {required_cutoff})")
    
    return result


def display_result(result, entrance_score, gpa, has_recommendation, 
                  category, extracurricular_score):
    """Format and display admission decision."""
    
    print("ADMISSION DECISION REPORT")
    print("=" * 60)
    
    # Input Summary
    print(f"Entrance Score: {entrance_score}")
    print(f"GPA: {gpa}")
    print(f"Recommendation: {has_recommendation}")
    print(f"Category: {category.upper()}")
    print(f"Extracurricular Score: {extracurricular_score}")
    
    # Bonus Details
    if result['bonuses_applied']:
        print(f"Bonus Applied: {' + '.join(result['bonuses_applied'])}")
        print(f"Effective Score: {result['effective_score']}")
    else:
        print("Bonus Applied: None")
    
    print("-" * 60)
    
    # Decision
    print(f"Result: {result['status']}")
    print(f"Reason: {result['reason']}")


def main():
    """Main admission system interface."""
    
    print("UNIVERSITY ADMISSION SCREENING SYSTEM")
    
    # Collect inputs with validation
    entrance_score = get_valid_input(
        "Entrance Score (0-100): ", 
        input_type=int, 
        min_val=0, 
        max_val=100
    )
    
    gpa = get_valid_input(
        "GPA (0-10): ", 
        input_type=float, 
        min_val=0, 
        max_val=10
    )
    
    has_recommendation = get_valid_input(
        "Recommendation (yes/no): ", 
        valid_options=['yes', 'no']
    )
    
    category = get_valid_input(
        "Category (general/obc/sc_st): ", 
        valid_options=['general', 'obc', 'sc_st']
    )
    
    extracurricular_score = get_valid_input(
        "Extracurricular Score (0-10): ", 
        input_type=int, 
        min_val=0, 
        max_val=10
    )
    
    # Process decision
    result = calculate_admission_decision(
        entrance_score, gpa, has_recommendation, 
        category, extracurricular_score
    )
    
    # Display output
    display_result(result, entrance_score, gpa, has_recommendation,
                  category, extracurricular_score)


if __name__ == "__main__":
    main()

UNIVERSITY ADMISSION SCREENING SYSTEM
ADMISSION DECISION REPORT
Entrance Score: 49
GPA: 9.43
Recommendation: yes
Category: GENERAL
Extracurricular Score: 9
Bonus Applied: Recommendation (+5) + Extracurricular (+3)
Effective Score: 57
------------------------------------------------------------
Result: WAITLISTED
Reason: Below GENERAL cutoff: Effective score 57 < 75 (required: 75)


Edge cases: generated by KIMI to test the constraints

In [6]:
def run_edge_case_tests():
    """Automated edge case validation with corrected expectations."""
    
    test_cases = [
        # Merit Rule Tests
        ("Merit boundary - exact 95", 95, 6.0, "no", "general", 0, "ADMITTED (Scholarship)"),
        ("Merit - above 95", 96, 4.0, "no", "general", 0, "ADMITTED (Scholarship)"),
        ("Merit - below 95 but high score", 94, 9.0, "no", "general", 0, "ADMITTED (Regular)"),  # FIXED
        ("Merit - below 95, below cutoff", 70, 9.0, "no", "general", 0, "WAITLISTED"),  # NEW
        
        # GPA Tests
        ("GPA boundary - exact 7.0", 70, 7.0, "no", "obc", 0, "ADMITTED (Regular)"),
        ("GPA boundary - below 7.0", 80, 6.9, "no", "general", 0, "REJECTED"),
        
        # Category Cutoff Tests
        ("General cutoff - exact 75", 75, 7.0, "no", "general", 0, "ADMITTED (Regular)"),
        ("General cutoff - below", 74, 9.0, "no", "general", 0, "WAITLISTED"),
        ("OBC cutoff - exact 65", 65, 7.0, "no", "obc", 0, "ADMITTED (Regular)"),
        ("OBC cutoff - below", 64, 9.0, "no", "obc", 0, "WAITLISTED"),
        ("SC/ST cutoff - exact 55", 55, 7.0, "no", "sc_st", 0, "ADMITTED (Regular)"),
        ("SC/ST cutoff - below", 54, 9.0, "no", "sc_st", 0, "WAITLISTED"),
        
        # Bonus Tests
        ("Both bonuses push to admit", 67, 8.0, "yes", "obc", 9, "ADMITTED (Regular)"),  # 67+8=75≥65
        ("Both bonuses but still fail", 60, 8.0, "yes", "general", 9, "WAITLISTED"),  # 60+8=68<75
        ("Only recommendation - reaches exact cutoff", 70, 8.0, "yes", "general", 5, "ADMITTED (Regular)"),  # 70+5=75, but wait... 75≥75 ADMIT!
        ("Only extracurricular", 72, 8.0, "no", "general", 9, "ADMITTED (Regular)"),  # 72+3=75≥75
        ("No bonuses - waitlisted", 74, 8.0, "no", "general", 5, "WAITLISTED"),
        
        # Edge Cases
        ("Zero entrance, SC/ST", 0, 7.0, "no", "sc_st", 0, "WAITLISTED"),
        ("All maximums", 100, 10.0, "yes", "general", 10, "ADMITTED (Scholarship)"),
        ("GPA fail trumps high score", 94, 6.5, "yes", "sc_st", 10, "REJECTED"),
    ]
    
    print("Running Edge Case Tests...")
    print("=" * 80)
    
    passed = 0
    failed = 0
    
    for desc, ent, gpa, rec, cat, extra, expected in test_cases:
        result = calculate_admission_decision(ent, gpa, rec, cat, extra)
        status = "PASS" if result['status'] == expected else "FAIL"
        
        if status == "PASS":
            passed += 1
        else:
            failed += 1
            
        print(f"{status}: {desc}")
        print(f"       Input: ent={ent}, gpa={gpa}, rec={rec}, cat={cat}, extra={extra}")
        print(f"       Expected: {expected}")
        print(f"       Got:      {result['status']}")
        if result['bonuses_applied']:
            print(f"       Bonuses:  {result['bonuses_applied']}")
            print(f"       Effective:{result['effective_score']}")
        print(f"       Reason:   {result['reason'][:70]}...")
        print()
    
    print("=" * 80)
    print(f"Results: {passed} passed, {failed} failed")


# Run the tests
run_edge_case_tests()

Running Edge Case Tests...
PASS: Merit boundary - exact 95
       Input: ent=95, gpa=6.0, rec=no, cat=general, extra=0
       Expected: ADMITTED (Scholarship)
       Got:      ADMITTED (Scholarship)
       Reason:   Auto-admitted: Entrance score 95 ≥ 95 (Merit Rule)...

PASS: Merit - above 95
       Input: ent=96, gpa=4.0, rec=no, cat=general, extra=0
       Expected: ADMITTED (Scholarship)
       Got:      ADMITTED (Scholarship)
       Reason:   Auto-admitted: Entrance score 96 ≥ 95 (Merit Rule)...

PASS: Merit - below 95 but high score
       Input: ent=94, gpa=9.0, rec=no, cat=general, extra=0
       Expected: ADMITTED (Regular)
       Got:      ADMITTED (Regular)
       Reason:   Meets GENERAL cutoff (94 ≥ 75) and GPA requirement (9.0 ≥ 7.0)...

PASS: Merit - below 95, below cutoff
       Input: ent=70, gpa=9.0, rec=no, cat=general, extra=0
       Expected: WAITLISTED
       Got:      WAITLISTED
       Reason:   Below GENERAL cutoff: Effective score 70 < 75 (required: 75)...

PASS: